# **Download and filesystem creation**

In [ ]:
# Create .kaggle directory and move kaggle.json
!mkdir -p ~/.kaggle
import os
kaggle_json = "/root/.kaggle/kaggle.json"
import shutil

if not os.path.exists(kaggle_json):

    from google.colab import files

    print("Upload your kaggle.json file:")
    uploaded = files.upload()

    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")

    os.makedirs("/root/.kaggle", exist_ok=True)

    shutil.move(
        "kaggle.json",
        kaggle_json
    )

    os.chmod(kaggle_json, 0o600)
# Verify the Kaggle configuration
!chmod 600 ~/.kaggle/kaggle.json
!ls -l ~/.kaggle

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
total 4
-rw------- 1 root root 72 Aug 21 13:36 kaggle.json


In [ ]:
!pip install kaggle

In [ ]:
import os

# List of Kaggle dataset identifiers (replace with your actual datasets)
kaggle_datasets = [
    "bhargavkumar1729/upload-folder-sdxl-aiface-dataset",
    "bhargavkumar1729/stylegan3-pytorch-faces-15k",
    "bhargavkumar1729/stylegan2-ada-pytorch-faces-15k",
    "bhargavkumar1729/gan-real-faces-50k-ffhq"
]

# Directory to store the downloaded data
download_dir = "./kaggle_data"
os.makedirs(download_dir, exist_ok=True)

for dataset_identifier in kaggle_datasets:
    print(f"\n--- Processing dataset: {dataset_identifier} ---")

    # Download the dataset
    # The -p flag specifies the path to download to
    # The -q flag makes the download quiet (no progress bar)
    download_command = f"kaggle datasets download -d {dataset_identifier} -p {download_dir} -q"
    print(f"Downloading: {dataset_identifier}")
    os.system(download_command)

    # Find the downloaded zip file(s)
    # Kaggle downloads the dataset as a zip file named after the dataset identifier's last part (e.g., 'first-dataset-name.zip')
    # or sometimes as a single zip file for the entire dataset.
    # We'll look for all .zip files in the download directory to be safe.

    # Get the base name of the dataset to look for its zip file
    dataset_base_name = dataset_identifier.split('/')[-1]
    zip_file_name_guess = os.path.join(download_dir, f"{dataset_base_name}.zip")

    # If the guessed zip file exists, unzip it. Otherwise, look for other zips.
    if os.path.exists(zip_file_name_guess):
        zip_files_to_process = [zip_file_name_guess]
    else:
        # Fallback: find all zip files in the download directory
        zip_files_to_process = [os.path.join(download_dir, f) for f in os.listdir(download_dir) if f.endswith('.zip')]

    if not zip_files_to_process:
        print(f"No zip files found for {dataset_identifier} in {download_dir}. Skipping unzip and delete.")
        continue

    for zip_file_path in zip_files_to_process:
        print(f"Unzipping: {os.path.basename(zip_file_path)}")
        # Unzip the file into the download directory
        unzip_command = f"unzip -o {zip_file_path} -d {download_dir}" # -o to overwrite existing files without prompting
        os.system(unzip_command)

        # Delete the zipped file
        print(f"Deleting zipped file: {os.path.basename(zip_file_path)}")
        os.remove(zip_file_path)

print("\nAll datasets processed.")
!rm -rf ./kaggle_data/my_dataset_upload
# Optional: List the contents of the download directory to verify
print("\nContents of the download directory:")
!ls -Fh {download_dir}



--- Processing dataset: bhargavkumar1729/upload-folder-sdxl-aiface-dataset ---
Downloading: bhargavkumar1729/upload-folder-sdxl-aiface-dataset
Unzipping: upload-folder-sdxl-aiface-dataset.zip
Deleting zipped file: upload-folder-sdxl-aiface-dataset.zip

--- Processing dataset: bhargavkumar1729/stylegan3-pytorch-faces-15k ---
Downloading: bhargavkumar1729/stylegan3-pytorch-faces-15k
Unzipping: stylegan3-pytorch-faces-15k.zip
Deleting zipped file: stylegan3-pytorch-faces-15k.zip

--- Processing dataset: bhargavkumar1729/stylegan2-ada-pytorch-faces-15k ---
Downloading: bhargavkumar1729/stylegan2-ada-pytorch-faces-15k
Unzipping: stylegan2-ada-pytorch-faces-15k.zip
Deleting zipped file: stylegan2-ada-pytorch-faces-15k.zip

--- Processing dataset: bhargavkumar1729/gan-real-faces-50k-ffhq ---
Downloading: bhargavkumar1729/gan-real-faces-50k-ffhq
Unzipping: gan-real-faces-50k-ffhq.zip
Deleting zipped file: gan-real-faces-50k-ffhq.zip

All datasets processed.

Contents of the download directory

In [ ]:
import os
import shutil
import random
from glob import glob
SEED = 42

random.seed(SEED)
# Define the base project directory
base_dir = "Faces_Project"
dataset_dir = os.path.join(base_dir, "Dataset")

# Define target subdirectories
train_real_dir = os.path.join(dataset_dir, "train", "real")
train_fake_dir = os.path.join(dataset_dir, "train", "fake")
val_real_dir = os.path.join(dataset_dir, "val", "real")
val_fake_dir = os.path.join(dataset_dir, "val", "fake")

# Create all necessary directories
os.makedirs(train_real_dir, exist_ok=True)
os.makedirs(train_fake_dir, exist_ok=True)
os.makedirs(val_real_dir, exist_ok=True)
os.makedirs(val_fake_dir, exist_ok=True)

print("Created directory structure:")
print(f"- {train_real_dir}")
print(f"- {train_fake_dir}")
print(f"- {val_real_dir}")
print(f"- {val_fake_dir}")

Created directory structure:
- Faces_Project/Dataset/train/real
- Faces_Project/Dataset/train/fake
- Faces_Project/Dataset/val/real
- Faces_Project/Dataset/val/fake


In [ ]:
# Source directory for real images
real_source_dir = "./kaggle_data/real"

# List all real image files (assuming .jpg, .png, etc.)
real_images = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    real_images.extend(glob(os.path.join(real_source_dir, '**', ext), recursive=True))

print(f"Found {len(real_images)} real images.")

# Shuffle the images
random.shuffle(real_images)

# Define split counts
train_real_count = 40000
val_real_count = 10000

if len(real_images) < (train_real_count + val_real_count):
    print(f"Warning: Not enough real images ({len(real_images)}) for desired split (50000). Using all available.")
    train_real_count = int(len(real_images) * 0.8) # Adjust proportionally
    val_real_count = len(real_images) - train_real_count

# Split into train and validation sets
train_real_images = real_images[:train_real_count]
val_real_images = real_images[train_real_count : train_real_count + val_real_count]

print(f"Moving {len(train_real_images)} real images to training.")
for i, img_path in enumerate(train_real_images):
    new_name = f"real_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(train_real_dir, new_name))

print(f"Moving {len(val_real_images)} real images to validation.")
for i, img_path in enumerate(val_real_images):
    new_name = f"real_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(val_real_dir, new_name))

print("Real images processed.")

Found 50000 real images.
Moving 40000 real images to training.
Moving 10000 real images to validation.
Real images processed.


In [ ]:
# Source directories for fake images
fake_source_dirs = [
    "./kaggle_data/Final_SDXL_AIface_DS",
    "./kaggle_data/stylegan2_faces",
    "./kaggle_data/stylegan3_faces"
]

# List all fake image files
fake_images = []
for source_dir in fake_source_dirs:
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        fake_images.extend(glob(os.path.join(source_dir, '**', ext), recursive=True))

print(f"Found {len(fake_images)} fake images.")

# Shuffle the images
random.shuffle(fake_images)

# Define split counts
train_fake_count = 40000
val_fake_count = 10000

if len(fake_images) < (train_fake_count + val_fake_count):
    print(f"Warning: Not enough fake images ({len(fake_images)}) for desired split (50000). Using all available.")
    train_fake_count = int(len(fake_images) * 0.8) # Adjust proportionally
    val_fake_count = len(fake_images) - train_fake_count

# Split into train and validation sets
train_fake_images = fake_images[:train_fake_count]
val_fake_images = fake_images[train_fake_count : train_fake_count + val_fake_count]

print(f"Moving {len(train_fake_images)} fake images to training.")
for i, img_path in enumerate(train_fake_images):
    new_name = f"fake_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(train_fake_dir, new_name))

print(f"Moving {len(val_fake_images)} fake images to validation.")
for i, img_path in enumerate(val_fake_images):
    new_name = f"fake_{i:05d}{os.path.splitext(img_path)[1]}"
    shutil.move(img_path, os.path.join(val_fake_dir, new_name))

print("Fake images processed.")

Found 50000 fake images.
Moving 40000 fake images to training.
Moving 10000 fake images to validation.
Fake images processed.


In [ ]:
# Remove the original downloaded data directory to save space
if os.path.exists("./kaggle_data"):
    shutil.rmtree("./kaggle_data")
    print("Removed original ./kaggle_data directory.")

print("\nVerifying final directory structure and counts:")
!ls -Fh {dataset_dir}
!echo ""
!ls -Fh {train_real_dir} | wc -l
!ls -Fh {train_fake_dir} | wc -l
!ls -Fh {val_real_dir} | wc -l
!ls -Fh {val_fake_dir} | wc -l

Removed original ./kaggle_data directory.

Verifying final directory structure and counts:
train/	val/

40000
40000
10000
10000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Model creation and training part**


sanity check

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.backends.cudnn.version())

True
12.8
91900


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# ============================================================
# NEW IMPORTS FOR CHECKPOINTING & RESUME
# ============================================================
import os
import csv
import random
import numpy as np
from google.colab import drive

# ============================================================
# MOUNT GOOGLE DRIVE (CRITICAL FOR PERSISTENCE)
# ============================================================
drive.mount('/content/drive',force_remount=True)

# ============================================================
# CONFIGURATION - THIS WORKS FOR ALL ACCOUNTS (NO EDITING NEEDED)
# ============================================================

PROJECT_ROOT = "/content/Faces_Project"
CHECKPOINT_DIR = f"/content/drive/MyDrive/Faces_Project/checkpoints"

DATASET_ROOT = f"{PROJECT_ROOT}/Dataset"

# Ensure the checkpoint directory exists (if running first time)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")
METRICS_PATH = os.path.join(CHECKPOINT_DIR, "training_metrics.csv")

BATCH_SIZE = 256
EPOCHS = 20
# ... rest of your config

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 4

# ============================================================
# VERIFY PATHS (SAFETY CHECK)
# ============================================================
if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f"Dataset not found at {DATASET_ROOT}. Did you mount Drive and set up the shared folder correctly?")

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Created new checkpoint directory: {CHECKPOINT_DIR}")

print(f"✅ Dataset path: {DATASET_ROOT}")
print(f"✅ Checkpoint path: {CHECKPOINT_DIR}")

# ============================================================
# FIXED SEED FOR REPRODUCIBILITY (CRUCIAL FOR RESUMING)
# ============================================================
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Optional: Ensure deterministic operations (slower but reproducible)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE")
print("=" * 60)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

# ============================================================
# TRAINING TRANSFORM
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (256, 256)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        5
    ),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

# ============================================================
# VALIDATION TRANSFORM
# ============================================================

val_transform = transforms.Compose([

    transforms.Resize(
        (256, 256)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

# ============================================================
# DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=f"{DATASET_ROOT}/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=f"{DATASET_ROOT}/val",
    transform=val_transform
)

print("\n" + "=" * 60)
print("DATASET")
print("=" * 60)

print(
    "Classes:",
    train_dataset.classes
)

print(
    "Class mapping:",
    train_dataset.class_to_idx
)

print(
    "Training images:",
    len(train_dataset)
)

print(
    "Validation images:",
    len(val_dataset)
)

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    worker_init_fn=lambda worker_id: set_seed(SEED + worker_id)  # Ensures dataloader shuffling is reproducible
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# ============================================================
# DEEP CUSTOM CNN
# ============================================================

class DeepCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            # BLOCK 1 (256 -> 128)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # BLOCK 2 (128 -> 64)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # BLOCK 3 (64 -> 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # BLOCK 4 (32 -> 16)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # BLOCK 5 (16 -> 8)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def save_checkpoint(epoch, model, optimizer, scheduler, scaler,
                    best_val_acc, best_f1,
                    rng_states, is_best=False):
    """Saves everything needed to resume training."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'best_val_acc': best_val_acc,
        'best_f1': best_f1,
        'rng_states': rng_states,
    }
    # Always save the latest
    torch.save(checkpoint, CHECKPOINT_PATH)

    # Save best model separately if requested
    if is_best:
        torch.save(checkpoint, BEST_CHECKPOINT_PATH)
        print(f"   >>> Best model saved! (Acc: {best_val_acc:.2f}%)")

torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

def load_checkpoint(model, optimizer, scheduler, scaler):
    checkpoint_path = "/content/drive/MyDrive/Faces_Project/checkpoints/latest_checkpoint.pth"

    try:
        # Try with weights_only=True first
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except Exception as e:
        print(f"⚠️ Falling back to weights_only=False (trusted source)")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    return checkpoint['epoch'], checkpoint['best_val_acc'], checkpoint['best_f1'], checkpoint.get('last_val_loss', 0.0)

def log_metrics(epoch, train_loss, train_acc, val_acc, precision, recall, f1, lr):
    """Appends epoch metrics to a CSV file in Google Drive."""
    file_exists = os.path.exists(METRICS_PATH)

    with open(METRICS_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_acc',
                             'precision', 'recall', 'f1', 'learning_rate'])
        writer.writerow([epoch+1, f"{train_loss:.4f}", f"{train_acc:.2f}",
                         f"{val_acc:.2f}", f"{precision:.4f}", f"{recall:.4f}",
                         f"{f1:.4f}", f"{lr:.7f}"])

# ============================================================
# CREATE MODEL
# ============================================================

model = DeepCNN().to(device)

# ============================================================
# MODEL INFORMATION
# ============================================================

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("MODEL")
print("=" * 60)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

# ============================================================
# LOSS, OPTIMIZER, SCHEDULER, MIXED PRECISION
# ============================================================

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# ============================================================
# TRY RESUME
# ============================================================

start_epoch, best_val_acc, best_f1, _ = load_checkpoint(
    model, optimizer, scheduler, scaler
)

# If resuming, we need to update the scheduler's step count to avoid resetting.
# CosineAnnealingLR needs to know how many steps were already taken.
# We'll just step it forward to catch up.
if start_epoch > 0:
    for _ in range(start_epoch):
        scheduler.step()
    print(f"   Scheduler caught up to epoch {start_epoch}.")
else:
    print("\n>>> Starting fresh training from epoch 0.")

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(start_epoch, EPOCHS):

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    model.train()
    total_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).float()
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()
    correct = 0
    total = 0
    TP = TN = FP = FN = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.float().to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                outputs = model(images)
                probabilities = torch.sigmoid(outputs)

            predictions = (probabilities >= 0.5).float().squeeze(1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            TP += ((predictions == 1) & (labels == 1)).sum().item()
            TN += ((predictions == 0) & (labels == 0)).sum().item()
            FP += ((predictions == 1) & (labels == 0)).sum().item()
            FN += ((predictions == 0) & (labels == 1)).sum().item()

    val_accuracy = 100 * correct / total
    precision = TP / (TP + FP) if TP + FP > 0 else 0.0
    recall = TP / (TP + FN) if TP + FN > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    # Update LR scheduler (step AFTER validation)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # TRACK BEST METRICS
    # --------------------------------------------------------
    is_best = False
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_f1 = f1
        is_best = True

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------
    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss:        {train_loss:.4f}")
    print(f"Train Accuracy:    {train_accuracy:.2f}%")
    print(f"Validation Acc:    {val_accuracy:.2f}%")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1 Score:          {f1:.4f}")
    print(f"Learning Rate:     {current_lr:.7f}")
    print(f"Best Val Acc so far: {best_val_acc:.2f}%")

    # --------------------------------------------------------
    # SAVE METRICS TO CSV
    # --------------------------------------------------------
    log_metrics(epoch, train_loss, train_accuracy, val_accuracy,
                precision, recall, f1, current_lr)

    # --------------------------------------------------------
    # SAVE CHECKPOINT (EVERY EPOCH)
    # --------------------------------------------------------
    rng_states = {
        'random': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        rng_states['cuda'] = torch.cuda.get_rng_state()

    save_checkpoint(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_val_acc=best_val_acc,
        best_f1=best_f1,
        rng_states=rng_states,
        is_best=is_best
    )

# ============================================================
# TRAINING COMPLETE
# ============================================================
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Final model saved to: {CHECKPOINT_PATH}")
print(f"Best model saved to: {BEST_CHECKPOINT_PATH}")
print(f"Metrics saved to: {METRICS_PATH}")

Mounted at /content/drive
✅ Dataset path: /content/Faces_Project/Dataset
✅ Checkpoint path: /content/drive/MyDrive/Faces_Project/checkpoints
DEVICE
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB

DATASET
Classes: ['fake', 'real']
Class mapping: {'fake': 0, 'real': 1}
Training images: 80000
Validation images: 20000

MODEL
Total parameters: 4,847,777
Trainable parameters: 4,847,777


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_662/1031021534.py:314: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use nump

⚠️ Falling back to weights_only=False (trusted source)
   Scheduler caught up to epoch 10.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
